<a href="https://colab.research.google.com/github/YoussefAli07/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Among pages with strong search visibility (high impressions, good ranking position), which ones are being overlooked by searchers despite that visibility, and can we systematically flag those as candidates for title/meta description rewrites?

Decision this supports: Instead of manually scanning thousands of content pages for something that might be wrong, FlyRank's content team can start with a ranked shortlist of pages that are already visible in search but silently failing to earn clicks — and decide, page by page, whether a title/meta rewrite or a different fix is worth their time.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Data:

This analysis uses the FlyRank ML Internship starter dataset: 30,000 rows, one row per content page, spread across 32 clients, with trailing 90-day performance metrics (impressions, clicks, CTR, and related fields) as of a single snapshot, not a time series.

Both content_id and client_id are pseudonyms, used only for grouping and train/test splitting (e.g., grouping by client to prevent the same client's pages appearing in both train and test) — never as model features.

What was excluded: Two columns, trend_direction and the label derived from it, were excluded from anything used as a model input, since both are computed from trend_pct — including them as features would leak the target into itself.

FlyRank also provides a much larger warehouse release (~79 million rows, spanning 17 months across 519,606 content items) which was explored early in the internship to understand FlyRank's data landscape, but was not used in this analysis, all results here come from the 30,000-row starter dataset only.

No client names, raw search queries, or identifying information appear in this dataset or anywhere in this paper. All fields are pre-anonymized by FlyRank.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

**Assumptions:** A page is worth manual review if it is (1) genuinely visible in search,
(2) not just stale/abandoned content, and (3) meaningfully underperforming its expected CTR
for its position.

**Baseline — rule-based gates:**
- `impressions_90d >= 81` → Visible (enough real search exposure to trust the CTR signal)
- `freshness_tier == '181+'` → excluded from the "stale" concern; flagged pages are not just
  neglected old content
- `ctr_gap <= -0.5` → Low CTR relative to expectation

**Score formula:** `ctr_gap.abs() * impressions_90d`, where `ctr_gap` is each page's CTR minus
the mean CTR for its `position_tier` (via `groupby('position_tier')['ctr'].transform('mean')`).
This rewards pages that are both badly underperforming and highly visible. This produced
**9 flagged pages** — all with `ctr = 0.0` despite real impressions, all in
`position_tier = page_1`.

**Label, features, model:**
- Label: `trend_pct`, clipped at the 1st/99th percentile in the training set only —
  necessary because an uncapped Decision Tree scored *worse than the naive baseline*
  (77.86 vs. 68.15 MAE). A handful of extreme outlier pages (trend_pct in the hundreds or
  thousands) were distorting the tree's splits away from accuracy on typical, everyday
  pages. Capping was not a preference but the difference between a useful model and a
  useless one. Test data was left uncapped, so the model is still evaluated against
  real-world values, not an artificially softened target.
- Features: `search_volume`, `competition`, `content_type`, `main_intent`, `word_count`
  (one-hot encoded) — never `trend_direction` or anything derived from it, since both are
  computed from `trend_pct` itself
- Model: Decision Tree Regressor (`max_depth=4`), chosen over a Random Forest — comparable MAE,
  but the shallow tree is directly traceable branch-by-branch, which matters when the reader
  making a decision from this work is a non-ML content strategist, not a data scientist

**Validation design & leakage checks:**
- Split via `GroupShuffleSplit` on `client_id`, not a plain random split, so pages from the same
  client never appear in both train and test
- A plain `train_test_split` (no client grouping) made the model look roughly **twice as good**
  as it actually was — a real, measured illustration of client leakage risk
- Confirmed `trend_direction` and its derived label were correctly excluded as features
- Stress-tested feature importance: `word_count` showed 98.4% importance in the shallow tree;
  a with/without test confirmed this is a structural artifact of a shallow, low-depth tree —
  not evidence of leakage
- Corrected the naive baseline to use the raw target mean, not a mean computed from the
  clipped/capped training data, avoiding an artificially weak comparison point

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Headline comparison (honest, client-grouped split):**

| Model | MAE | Improvement over baseline |
|---|---|---|
| Naive baseline (predict training mean) | 68.15 | — |
| Decision Tree (max_depth=4, capped training) | 65.24 | 4.3% |
| Random Forest (100 trees, capped training) | 65.14 | 4.4% |

Both models beat the naive baseline, but only modestly. The Random Forest's added
complexity bought essentially no improvement (0.1 MAE, likely noise) over the single
Decision Tree, reinforcing the interpretable model as the right choice.

**Why the split method matters:** a plain random split (no client grouping) makes the
same model look nearly twice as good as it really is:

| Split type | Naive MAE | Model MAE | Improvement |
|---|---|---|---|
| Honest (grouped by client_id) | 68.15 | 65.24 | 4.3% |
| Dishonest (random, no grouping) | 69.44 | 63.46 | 8.6% |

The 4.3% figure is the one that reflects real-world deployment (performance on clients
the model has never seen) and is the number reported as this project's result. The 8.6%
figure is reported here only to demonstrate why the grouped split was necessary.

**Precision@K comparison:** attempted between the ML-07 rule and the model on the held-out
test split, but the rule's three gates flagged 0 pages within this client subset (consistent
with flagging only 9 pages across the full 30,000-row dataset). A direct precision@K
comparison was therefore not meaningful on this split, not a failure of either method, but
a mismatch between the rule's intentional selectivity and a single train/test partition's
scale.

**Error analysis:** the model's largest errors occur on pages with extreme real `trend_pct`
values, an expected consequence of capping the training target at the 99th percentile. A
second, more instructive error type appeared on an ordinary page that received a wildly high
prediction, traced to the model's heavy reliance on `word_count`, confirmed in validation as
a shallow-tree structural artifact (removing it changed MAE by less than 1%), not genuine
predictive signal.

## 5. Limitations

*What this work cannot claim.*

## Limitations

**Feature importance is misleading, not a real signal:** the Decision Tree assigns 98.4%
of its feature importance to `word_count`, but a controlled with/without test (ML-09)
showed removing it barely changes MAE (65.24 → 64.90). This importance score reflects a
structural artifact of a shallow (max_depth=4) tree locking onto one early split — it
should not be read as "word count drives trend performance," and should not inform content
decisions on its own.

**Capping limits prediction on extreme cases:** training-target capping (1st/99th
percentile) was necessary to make the model useful at all (an uncapped model scored worse
than the naive baseline), but it means the model cannot meaningfully predict pages with
genuinely extreme trend_pct values, it will underpredict rare viral or collapse cases by
design, a known and accepted tradeoff.

**Precision@K comparison was inconclusive:** the ML-07 rule's strict gates flagged 0 pages
within the held-out test split's client subset, making a direct rule-vs-model precision
comparison impossible on this data. This reflects the rule's intentional selectivity at
small scale, not a failure of either method.

**The flagged shortlist is narrow, not representative.** All 9 flagged pages share the same
`content_type` (keyword article) and `position_tier` (page_1), and over half (5 of 9) belong
to a single client. The rule has not been validated on other content types, position tiers,
or a broader client base — a different mix of pages could show a different pattern entirely,
or none at all.

**Sample size and generalizability:** this entire analysis is built on a 30,000-row, 32-client
snapshot — a small slice of FlyRank's full warehouse (~520,000 content items across roughly
100 clients). Patterns confirmed here are real *within this sample*, but haven't been tested
against the full client base; a page pattern common in this subset could be rare, or a
different pattern entirely could dominate, across FlyRank's complete dataset.

**Detection, not prescription:** this work flags pages worth reviewing — it does not test
or prove that title/meta description rewrites (or any other fix) will recover engagement.
The 9 flagged pages are candidates for review, not a guaranteed list of fixable pages. Any
actual improvement depends on decisions and changes made by FlyRank's content team, which
fall outside what this analysis measured.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked recommendations

Nine pages passed all three ML-07 gates (Visible, Stale, Low CTR) in the 30,000-row
snapshot. All nine have `ctr = 0.0`. Zero clicks despite real search impressions. Ranked
by score (|ctr_gap| × impressions_90d), highest priority first:

| Rank | Impressions (90d) | Reason | Action |
|---|---|---|---|
| 1 | 429 | Visible, Stale, Low CTR | Needs review — potential title/meta rewrite |
| 2 | 265 | Visible, Stale, Low CTR | Needs review — potential title/meta rewrite |
| 3 | 198 | Visible, Stale, Low CTR | Needs review — potential title/meta rewrite |
| ... | ... | ... | ... |
| 9 | 81 | Visible, Stale, Low CTR | Needs review — potential title/meta rewrite |

The ranking reflects wasted opportunity, not just failure: a page with 429 impressions and
zero clicks represents more lost visibility than a page with 81 impressions and zero clicks,
even though both trip the same three gates.

**Before acting on any flagged page:** confirm the page still matches this profile (data is
a trailing-90-day snapshot; situations may have shifted), read the page against its target
query (a mismatched intent needs a content rewrite, not a title tweak), and rule out
non-title causes for zero clicks — bot traffic, tracking issues, or a SERP snippet problem
can produce the identical pattern. No rewrite should be auto-published without human review,
and no page should be promised a specific traffic improvement — the score reflects
association, not a guaranteed causal effect.

**Most important signal to watch:** a sharp change in the flagged count (currently 9) on a
rerun. Because `ctr_gap` is computed relative to the mean CTR within a page's position tier,
it is not a fixed measurement, it shifts whenever the underlying population shifts (new
clients onboarded, content mix changes, seasonal swings). A sudden jump to 0 or to hundreds
of flagged pages is a signal to re-examine the gate thresholds, not evidence the client base
improved or worsened overnight. Other signals worth watching: the 81-impression threshold
losing meaning if overall traffic shifts significantly, a page that's rewritten but stays
flagged weeks later, and a client's content mix moving away from the keyword-article pages
this rule was validated on.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


1. **Model vs. baseline MAE** — bar chart: Naive baseline (68.15) vs. Decision Tree (65.24)
   vs. Random Forest (65.14). Caption: "A shallow, interpretable model modestly but
   genuinely beats the naive baseline."

2. **Honest vs. dishonest split comparison** — bar chart: 4.3% (grouped split) vs. 8.6%
   (random split) improvement over baseline. Caption: "A random split without client
   grouping makes the same model look nearly twice as good as it really is."

3. **Feature importance, paired with the leakage check** — side-by-side bar chart:
   raw importance (word_count 98.4%, search_volume 1.5%, others ~0) next to the
   with/without MAE test (65.24 with word_count vs. 64.90 without). Caption: "High
   importance does not mean high real impact, removing word_count barely changes
   accuracy."

4. **The 9-page ranked shortlist** — clean table (already built in Recommendations),
   reused as-is; no chart needed since a ranked list is inherently table-shaped.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
